In [58]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import time, timedelta
import re


Загружаем данные и сразу присваиваем столбцам ID и Contact Name - тип str

In [59]:
DATA_RAW = Path('../data/raw')
DATA_PROCESSED = Path('../data/processed')

deals = pd.read_excel(DATA_RAW / 'Deals (Done).xlsx', dtype={'Id':str, 'Contact Name': str})
deals['Created Time'] = pd.to_datetime(deals['Created Time'], format='%d.%m.%Y %H:%M')
deals['Closing Date'] = pd.to_datetime(deals['Closing Date'], format='%d.%m.%Y')
print(f"Исходно: {deals.shape}")

Исходно: (21595, 23)


Первичный осмотр по типам, фильтрам и пропускам был проведен в Excel.
Поэтому тут проверяем количество пропусков в датасете.

In [60]:
deals.shape #количество строк и столбцов изначальное (21595, 23)

(21595, 23)

In [61]:
deals.dtypes #типы в столбцах

Id                                str
Deal Owner Name                   str
Closing Date           datetime64[us]
Quality                           str
Stage                             str
Lost Reason                       str
Page                              str
Campaign                          str
SLA                            object
Content                           str
Term                              str
Source                            str
Payment Type                      str
Product                           str
Education Type                    str
Created Time           datetime64[us]
Course duration               float64
Months of study               float64
Initial Amount Paid            object
Offer Total Amount             object
Contact Name                      str
City                              str
Level of Deutsch               object
dtype: object

In [62]:
deals.isnull().sum() #количество пропусков

Id                         2
Deal Owner Name           31
Closing Date            6950
Quality                 2255
Stage                      2
Lost Reason             5471
Page                       2
Campaign                5528
SLA                     6062
Content                 7448
Term                    9141
Source                     2
Payment Type           21099
Product                18003
Education Type         18295
Created Time               2
Course duration        18008
Months of study        20755
Initial Amount Paid    17430
Offer Total Amount     17410
Contact Name              63
City                   19084
Level of Deutsch       20344
dtype: int64

По методике - пройтись по каждому столбцу, заполнить пропуски по возможности, удалить мусор и дубликаты, поменять типы. 

In [63]:
deals.nunique() #уникальные значения по каждому столбцу

Id                     21593
Deal Owner Name           27
Closing Date             359
Quality                    6
Stage                     13
Lost Reason               21
Page                      34
Campaign                 154
SLA                    13357
Content                  187
Term                     220
Source                    13
Payment Type               3
Product                    5
Education Type             3
Created Time           20334
Course duration            2
Months of study           12
Initial Amount Paid       24
Offer Total Amount        21
Contact Name           18089
City                     876
Level of Deutsch         215
dtype: int64

Общая картинка - по пропускам
в Id пропусков 2, остальные значения уникальные,
Deal Owner Name - 27 менеджеров и 31 пропуск - можем заполнить Unknown manager
Closing Date 6950 пропусков =  незакрытых сделок из 21593 клиентов (а это около 68% закрытых сделок!! больше половины)
Quality                 2255 - оставляем пропуски как есть
Stage                      2 - два пропуска по стадии - можем посмотреть - заполнить если оплачено, но в целом не существенно
Lost Reason             5471 - пока опускаем
Page                       2 - пропускаем, статистически не значимо по сравнению с основным объемом
Campaign                5528 - пропускаем
SLA                     6062 - пропускаем - можем посмотреть потом может мы можем его как-то посчитать для оценки эффектичности менеджеров, например Created time - дата подачи заявки

Content                 7448 - пропускаем
Term                    9141 -  пропускаем
Source                     2 - пропускаем, статистически не значимо
Payment Type           21099 - можно закрыть часть - по значку закрытия сделки и сумме оплаты
Product                18003 - можно посмотреть если ли корреляция с суммлй оплаты если она есть
Education Type         18295 - посмотреть есть ли корреляция с Offer Total Amount
Created Time               2 - пропускаем, статистически не значимо
Course duration        18008 - заполняем частично в зависимости от продукта и Education Type 
Months of study        20755 - оставляем как есть
Initial Amount Paid    17430 - зпаолняем для тех что с Closing Date - по Months of study и Offer Total Amount. Переводим в ЕВРО
Offer Total Amount     17410  - переводим в Евро, заполняем для тех что с Closing Date в зависимости от Product и Education Type		

Contact Name              63 - это мы не восстановим. Но видим что контактов меньше чем заявок, что в целом не понятно пока что
City                   19084 - пропускаем
Level of Deutsch       20344 - пропускаем

In [64]:
# Сколько строк, где ВСЕ значения NaN
fully_empty = deals.isna().all(axis=1).sum()
print(f"Полностью пустых строк: {fully_empty}")

Полностью пустых строк: 1


In [65]:
deals[deals['Id'].isnull()]  # у нас явные две полностью пустые строчки - удаляем

,Id,Deal Owner Name,Closing Date,Quality,Stage,Lost Reason,Page,Campaign,SLA,Content,...,Product,Education Type,Created Time,Course duration,Months of study,Initial Amount Paid,Offer Total Amount,Contact Name,City,Level of Deutsch
21593,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN
21594,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,#REF!,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [66]:
deals = deals.dropna(subset=['Id']) #удаляем их. Изначалльно строк (21595, 23)
deals.shape

(21593, 23)

In [67]:
deals.isnull().sum() #количество пропусков, таблицв для обновления

Id                         0
Deal Owner Name           29
Closing Date            6948
Quality                 2253
Stage                      0
Lost Reason             5469
Page                       0
Campaign                5526
SLA                     6060
Content                 7446
Term                    9139
Source                     0
Payment Type           21097
Product                18001
Education Type         18294
Created Time               0
Course duration        18006
Months of study        20753
Initial Amount Paid    17428
Offer Total Amount     17408
Contact Name              61
City                   19082
Level of Deutsch       20342
dtype: int64

#2 -  Замена типов данных
Закончу с типами данных - самыее главные остались Initial Amount Paid   и  Offer Total Amount -там надо убрать значки евро или доллара и перевести все во float

In [68]:
deals['Initial Amount Paid'].astype(str).value_counts().head(50)

Initial Amount Paid
1000          2623
0              876
300            188
500             94
350             82
2000            58
11000           38
200             31
11500           25
3500            22
€ 3.500,00      16
1500            16
450             16
5000            14
4000            13
100             12
3000            11
4500            10
400             10
1                3
600              3
1200             2
700              1
9                1
Name: count, dtype: int64

In [69]:
deals['Offer Total Amount'].astype(str).value_counts().head(30)

Offer Total Amount
11000         1860
0              848
11500          394
5000           295
4000           252
3500           133
9000           115
2500            70
2000            63
3000            58
4500            57
€ 2.900,00      20
1200             6
1000             3
1500             3
10000            2
1                2
6500             1
€ 11398,00       1
11111            1
6000             1
Name: count, dtype: int64

Видим, что у нас есть знак евро и точки в данных. Напишем для них функцию и применим для обоих столбцов.

In [70]:
def normalize_eur(series: pd.Series) -> pd.Series:
    """Преобразует строки вида '€ 3.500,00' и '1000' в float (евро)."""
    cleaned = (
        series.astype(str)
        .str.replace('€', '', regex=False)
        .str.replace(' ', '', regex=False)
        .str.replace('.', '', regex=False)
        .str.replace(',', '.', regex=False)
    )
    return pd.to_numeric(cleaned, errors='coerce')

deals['Initial Amount Paid'] = normalize_eur(deals['Initial Amount Paid'])
deals['Offer Total Amount'] = normalize_eur(deals['Offer Total Amount'])

In [71]:
deals['Offer Total Amount'].astype(str).value_counts().head(30)

Offer Total Amount
11000.0    1860
0.0         848
11500.0     394
5000.0      295
4000.0      252
3500.0      133
9000.0      115
2500.0       70
2000.0       63
3000.0       58
4500.0       57
2900.0       20
1200.0        6
1000.0        3
1500.0        3
10000.0       2
1.0           2
6500.0        1
11398.0       1
11111.0       1
6000.0        1
Name: count, dtype: int64

In [72]:
deals[['Initial Amount Paid', 'Offer Total Amount']].dtypes #проверка

Initial Amount Paid    float64
Offer Total Amount     float64
dtype: object

Также SLA имеет тип object - поменяем его

In [73]:
# Какой тип у каждого значения?
print(deals['SLA'].apply(type).value_counts())

SLA
<class 'datetime.time'>         13672
<class 'float'>                  6060
<class 'datetime.timedelta'>     1861
Name: count, dtype: int64


мы видим три разных типа данных в колонке, преобразуем оба типа к единому `timedelta`, после чего применить `pd.to_timedelta` для финальной унификации.

In [74]:
def to_timedelta_safe(value):
    if pd.isna(value):
        return pd.NaT
    if isinstance(value, time):
        return timedelta(hours=value.hour, minutes=value.minute, seconds=value.second)
    if isinstance(value, timedelta):
        return value
    return pd.NaT  # для всех неожиданных случаев

In [75]:
deals['SLA'] = deals['SLA'].apply(to_timedelta_safe)
deals['SLA'] = pd.to_timedelta(deals['SLA'])  # финальная унификация в pandas Timedelta

In [76]:
print(deals['SLA'].dtype)
print(deals['SLA'].describe())
print(f"NaN: {deals['SLA'].isna().sum()}")

timedelta64[us]
count                     15533
mean     1 days 08:10:26.223330
std      8 days 12:47:32.635614
min             0 days 00:00:03
25%             0 days 01:13:00
50%             0 days 05:31:34
75%             0 days 15:38:38
max           311 days 10:34:24
Name: SLA, dtype: object
NaN: 6060


Краткие выводы - минимум - есть звонки когда реагируют мгновенно после заявки - через 3 секунды, а есть 311 дней - возможно это выброс - посмотреть паотом при работе с выбросами. 

In [77]:
deals.dtypes #типы в столбцах финальные - все ок.

Id                                 str
Deal Owner Name                    str
Closing Date            datetime64[us]
Quality                            str
Stage                              str
Lost Reason                        str
Page                               str
Campaign                           str
SLA                    timedelta64[us]
Content                            str
Term                               str
Source                             str
Payment Type                       str
Product                            str
Education Type                     str
Created Time            datetime64[us]
Course duration                float64
Months of study                float64
Initial Amount Paid            float64
Offer Total Amount             float64
Contact Name                       str
City                               str
Level of Deutsch                object
dtype: object

In [78]:
print(deals['Level of Deutsch'].dtype)
print(deals['Level of Deutsch'].value_counts(dropna=False).head(20))

object
Level of Deutsch
NaN      20342
B1         219
б1         118
в1         100
b1          93
Б1          93
В1          63
А2          53
B2          45
а2          32
б2          30
Б2          20
в2          20
в1-в2       19
b2          19
A2          18
?           15
В2          14
с1          13
А1          11
Name: count, dtype: int64


# 3 - КАТЕГОРИАЛЬНАЯ ЧИСТКА

посмотрим все заголовки и уникальные значения для данного датасета

In [79]:
categorical_cols = [
    'Stage', 'Quality', 'Lost Reason',
    'Source', 'Campaign', 'Product',
    'Education Type', 'Payment Type',
    'City', 'Level of Deutsch',
    'Deal Owner Name'
]

for col in categorical_cols:
    print(f"\n=== {col} ===")
    print(deals[col].value_counts(dropna=False))


=== Stage ===
Stage
Lost                         15743
Call Delayed                  2248
Registered on Webinar         2072
Payment Done                   858
Waiting For Payment            325
Qualificated                   128
Registered on Offline Day      100
Need to Call - Sales            33
Need To Call                    31
Test Sent                       25
Need a consultation             23
New Lead                         6
Free Education                   1
Name: count, dtype: int64

=== Quality ===
Quality
E - Non Qualified    7634
D - Non Target       6248
C - Low              3459
NaN                  2253
B - Medium           1564
A - High              432
F                       3
Name: count, dtype: int64

=== Lost Reason ===
Lost Reason
NaN                                        5469
Doesn't Answer                             4135
Changed Decision                           2146
Duplicate                                  1771
Non target                              

**Stage (статусы воронки):** 13 уникальных значений. Будет сгруппировано в три категории: Won (Payment Done, 858), Lost (Lost, 15743), In Progress (остальные ~5000). Базовая конверсия Won/(Won+Lost) = **5.17%**.

**Level of Deutsch:** колонка содержит 216 уникальных значений на 1251 заполненных строк.Приведем все к одному уровню через regex. **94% пропусков** — анализ "влияние немецкого на конверсию" будет иметь индикативный характер.

**Source:** обнаружено 159 строк с источником .`Test` — надо проверить что за линформация там, возможно тестовые и можно удалить. 

**City:** 348 строк со значением `-` — закодированный пропуск, заменим на NaN.

**Quality:** 3 строки со значением `F` (вероятно, опечатка или устаревшая категория) — заменим на NaN.

**Lost Reason:** 22 уникальные причины потери сделки. Группируются: проблемы с контактом (~9000), возражения по продукту (~2500), несоответствие клиента (~1000), отказ от Gutschein и финансовые причины (~300). Богатый материал для рекомендаций бизнесу.

In [80]:
#Заменим в City прочерк на Nan
deals['City'] = deals['City'].replace('-', np.nan)

In [81]:
deals['Quality'] = deals['Quality'].replace('F', np.nan)


In [82]:
categorical_cols = ['City', 'Quality']

for col in categorical_cols:
      print(deals[col].value_counts(dropna=False))

City
NaN                19430
Berlin               182
München               74
Hamburg               62
Nürnberg              45
                   ...  
Bad Gandersheim        1
Braubach               1
Geldern                1
Bassum                 1
Brake                  1
Name: count, Length: 876, dtype: int64
Quality
E - Non Qualified    7634
D - Non Target       6248
C - Low              3459
NaN                  2256
B - Medium           1564
A - High              432
Name: count, dtype: int64


In [83]:
test_leads = deals[deals['Source'] == 'Test']
print(f"Всего лидов с Source=Test: {len(test_leads)}")
print(f"\nРаспределение по Stage:")
print(test_leads['Stage'].value_counts(dropna=False))

Всего лидов с Source=Test: 159

Распределение по Stage:
Stage
Lost                    81
Call Delayed            73
Payment Done             3
Need to Call - Sales     1
Waiting For Payment      1
Name: count, dtype: int64


Вывод - Source Тест - не похожи на тестовые значения, значит удалять не будем.

Разделим категории Stage на три категории - Won — Payment Done, Lost — Lost, In Progress. Но у нас в части данных с Payment done - нет информации о сумме платежа и курсе и дате закрытия. Сделаем диагностику. 

In [116]:
won = deals[deals['Stage'] == 'Payment Done']
print(f"Всего Payment Done: {len(won)}")
print()
print(f"Без Closing Date: {won['Closing Date'].isna().sum()}")
print(f"Без Initial Amount Paid: {won['Initial Amount Paid'].isna().sum()}")
print(f"Без Product: {won['Product'].isna().sum()}")
print(f"Без Months of study: {won['Months of study'].isna().sum()}")
print(f"Без Payment Type: {won['Payment Type'].isna().sum()}")

Всего Payment Done: 858

Без Closing Date: 337
Без Initial Amount Paid: 15
Без Product: 17
Без Months of study: 18
Без Payment Type: 494


Вывод - у нас 18 строк с людьми которые не начали учиться - и 15 строк без суммы - от целого числа это менее 2%. Тем не менее, информацию не удаляем, 
для дальшейшей работы, чтобы оставить только чистые данные, сделаем флаг при котором есть статус Payment Done и Initial Amount Paid > 0.

In [117]:
stage_mapping = {  #разделяем категории на Won — Payment Done, Lost — Lost, In Progress
    'Payment Done': 'Won',
    'Lost': 'Lost',
    'Call Delayed': 'In Progress',
    'Registered on Webinar': 'In Progress',
    'Waiting For Payment': 'In Progress',
    'Qualificated': 'In Progress',
    'Registered on Offline Day': 'In Progress',
    'Need to Call - Sales': 'In Progress',
    'Need To Call': 'In Progress',
    'Test Sent': 'In Progress',
    'Need a consultation': 'In Progress',
    'New Lead': 'In Progress',
    'Free Education': 'Won'
}

deals['Stage_Group'] = deals['Stage'].map(stage_mapping)

In [118]:
# Финансовое определение Won: Stage = Payment Done И есть реальная оплата
deals['is_won_confirmed'] = (
    (deals['Stage'] == 'Payment Done')
    & deals['Initial Amount Paid'].notna()
    & (deals['Initial Amount Paid'] > 0)
)
print(f"Won по Stage: {(deals['Stage_Group']=='Won').sum()}")
print(f"Won Confirmed (с оплатой > 0): {deals['is_won_confirmed'].sum()}")
print(f"Разница: {(deals['Stage_Group']=='Won').sum() - deals['is_won_confirmed'].sum()} сделок без подтверждённой оплаты")

Won по Stage: 859
Won Confirmed (с оплатой > 0): 840
Разница: 19 сделок без подтверждённой оплаты


859 строк - так как добавили в won 1 строку Free Education.

In [119]:
print(deals['Stage_Group'].value_counts(dropna=False))

Stage_Group
Lost           15743
In Progress     4991
Won              859
Name: count, dtype: int64


# !!!!!!

In [120]:
print(deals['Stage_Group'].isna().sum())

0


Также у нас выделяются три основынх продукта за которые люди платят - исходя из фильтров в Экселе, сделаю для них флаг

In [121]:
MAIN_PRODUCTS = ['Web Developer', 'Digital Marketing', 'UX/UI Design']
deals['is_main_product'] = deals['Product'].isin(MAIN_PRODUCTS)
print(f"Сделок с основным продуктом: {deals['is_main_product'].sum()}")
print(f"\nРаспределение Product:")
print(deals['Product'].value_counts(dropna=False))

Сделок с основным продуктом: 3587

Распределение Product:
Product
NaN                    18001
Digital Marketing       1990
UX/UI Design            1022
Web Developer            575
Find yourself in IT        4
Data Analytics             1
Name: count, dtype: int64


### Приведение к одной системе немецкого языка 

In [88]:
# Паттерн: одна буква (лат A/B/C или кир А/В/С/Б) + цифра 1 или 2

pattern = re.compile(r'([AaBbCcАаВвСсБб])\s*([12])')
translit = {
    'А': 'A', 'а': 'A',  # кир А → лат A
    'В': 'B', 'в': 'B',  # кир В → лат B (визуально похожи!)
    'Б': 'B', 'б': 'B',  # кир Б → лат B (фонетически)
    'С': 'C', 'с': 'C',  # кир С → лат C
}

In [89]:
def extract_deutsch_level(value):
    if pd.isna(value):
        return None
    s = str(value)
    match = pattern.search(s)
    if not match:
        return None
    letter, digit = match.group(1), match.group(2)
    # Транслитерация если кириллица
    letter = translit.get(letter, letter)
    return f"{letter.upper()}{digit}"

deals['Level of Deutsch'] = deals['Level of Deutsch'].apply(extract_deutsch_level)

pattern.search(s) вернёт первое совпадение в строке. Это значит:

"B1-B2" → возьмётся B1 (нижний уровень)
"А2-В1 учит" → возьмётся A2 (нижний)
"В1 (учится на В2 уже)" → возьмётся B1 (текущий, не желаемый)
"В январе - В2 сдает" → возьмётся B2 (дату не парсим, но повезло)
"точно уровень не знаю..." → ничего не найдено → NaN

In [90]:
print(deals['Level of Deutsch'].value_counts(dropna=False))
print(f"\nТипов значений: {deals['Level of Deutsch'].nunique()}")

Level of Deutsch
NaN    20400
B1       817
B2       171
A2       149
C1        27
A1        26
C2         3
Name: count, dtype: int64

Типов значений: 6


Потеряно 58 значений (1251 → 1193) — это  58 строк со свободным текстом без явного уровня.

In [91]:
deals['Deal Owner Name'] = deals['Deal Owner Name'].fillna('Unknown manager') # для незаполненных колонок менеджера - неизхвестный менеджер

#  Проверка на дубликаты

In [92]:
print(f"Дубликатов по Id: {deals['Id'].duplicated().sum()}")
print(f"Полных дубликатов строк: {deals.duplicated().sum()}")

Дубликатов по Id: 0
Полных дубликатов строк: 0


In [93]:
# Сколько контактов имеют сколько сделок
contact_counts = deals['Contact Name'].value_counts(dropna=False)
print(f"Распределение количества сделок на контакт:")
print(contact_counts.value_counts().sort_index().head(15))

Распределение количества сделок на контакт:
count
1     15551
2      1990
3       385
4       113
5        28
6         9
7         4
8         2
10        2
11        1
13        1
19        1
39        1
54        1
61        1
Name: count, dtype: int64


In [94]:
# Топ-10 контактов с наибольшим количеством сделок
top_repeats = contact_counts.head(10)
print("Топ-10 контактов с наибольшим количеством сделок:")
print(top_repeats)

Топ-10 контактов с наибольшим количеством сделок:
Contact Name
NaN                    61
5805028000003014152    54
5805028000005448163    39
5805028000017522090    19
5805028000014478367    13
5805028000000872003    11
5805028000002150769    10
5805028000003966221    10
5805028000003456746     8
5805028000005124061     8
Name: count, dtype: int64


In [95]:
# Контакты, у которых больше одной оплаченной сделки (Stage_Group = Won)
won_per_contact = deals[deals['Stage_Group'] == 'Won']['Contact Name'].value_counts()
print(f"\nКонтактов с >1 оплаченной сделкой:")
print((won_per_contact > 1).sum())

print(f"\nКонтактов с 2+ Won:")
print(won_per_contact[won_per_contact > 1].head(20))


Контактов с >1 оплаченной сделкой:
4

Контактов с 2+ Won:
Contact Name
5805028000043599528    2
5805028000009072093    2
5805028000033276868    2
5805028000017756471    2
Name: count, dtype: int64


In [96]:
# Что это за контакт с 54 сделками?
suspicious_contact = '5805028000003014152'
investigation = deals[deals['Contact Name'] == suspicious_contact]
print(f"Сделок: {len(investigation)}")
print(f"\nDeal Owners: ")
print(investigation['Deal Owner Name'].value_counts())
print(f"\nStage:")
print(investigation['Stage'].value_counts())
print(f"\nSource:")
print(investigation['Source'].value_counts())
print(f"\nДиапазон Created Time:")
print(f"  с {investigation['Created Time'].min()}")
print(f"  по {investigation['Created Time'].max()}")

Сделок: 54

Deal Owners: 
Deal Owner Name
Julia Nelson     39
Diana Evans       8
Ian Miller        3
Bob Brown         2
Kevin Parker      1
Charlie Davis     1
Name: count, dtype: int64

Stage:
Stage
Lost    54
Name: count, dtype: int64

Source:
Source
Google Ads     31
Youtube Ads    19
CRM             2
Tiktok Ads      2
Name: count, dtype: int64

Диапазон Created Time:
  с 2023-07-25 11:05:00
  по 2024-05-30 22:26:00


сколько сделок сдделали контакты без имени?

In [97]:
no_contact = deals[deals['Contact Name'].isna()]
print(no_contact['Stage_Group'].value_counts())

Stage_Group
Lost           43
Won            10
In Progress     8
Name: count, dtype: int64


In [98]:
# Какие ещё контакты с 6-54 
mid_repeats = contact_counts[(contact_counts >= 6) & (contact_counts <= 54)].index
print(f"Контактов с 6-54 сделками: {len(mid_repeats)}")
print(f"Всего сделок у них: {deals['Contact Name'].isin(mid_repeats).sum()}")

# Что у них со стадиями?
mid_repeat_deals = deals[deals['Contact Name'].isin(mid_repeats)]
print(f"\nИх Stage_Group:")
print(mid_repeat_deals['Stage_Group'].value_counts())

Контактов с 6-54 сделками: 22
Всего сделок у них: 254

Их Stage_Group:
Stage_Group
Lost           190
In Progress     61
Won              3
Name: count, dtype: int64


In [99]:
deals['is_duplicate_lead'] = deals['Lost Reason'] == 'Duplicate'
print(f"Дублирующих лидов: {deals['is_duplicate_lead'].sum()}")
print(f"Доля от Lost: {deals['is_duplicate_lead'].sum() / (deals['Stage_Group'] == 'Lost').sum() * 100:.1f}%")

Дублирующих лидов: 1771
Доля от Lost: 11.2%


In [100]:
# Outlier = больше 5 сделок И ВСЕ Lost (т.е. ни одной Won, ни одной In Progress на закрытие)
contact_stats = deals.groupby('Contact Name').agg(
    total_deals=('Id', 'count'),
    won_deals=('Stage_Group', lambda x: (x == 'Won').sum())
)

outlier_contacts = contact_stats[
    (contact_stats['total_deals'] > 5) & 
    (contact_stats['won_deals'] == 0)
].index

deals['is_outlier_contact'] = deals['Contact Name'].isin(outlier_contacts)
print(f"Outlier контактов: {len(outlier_contacts)}")
print(f"Помечено сделок: {deals['is_outlier_contact'].sum()}")

Outlier контактов: 19
Помечено сделок: 199


In [101]:
print(deals.shape)
print(deals.dtypes)

(21593, 26)
Id                                 str
Deal Owner Name                    str
Closing Date            datetime64[us]
Quality                            str
Stage                              str
Lost Reason                        str
Page                               str
Campaign                           str
SLA                    timedelta64[us]
Content                            str
Term                               str
Source                             str
Payment Type                       str
Product                            str
Education Type                     str
Created Time            datetime64[us]
Course duration                float64
Months of study                float64
Initial Amount Paid            float64
Offer Total Amount             float64
Contact Name                       str
City                               str
Level of Deutsch                   str
Stage_Group                        str
is_duplicate_lead                 bool
is_outlier_co

In [102]:
deals.shape


(21593, 26)

## Проверка по диапазонам дат

In [103]:
print("=" * 60)
print("ФИНАЛЬНАЯ СВОДКА — DEALS")
print("=" * 60)
print(f"Размер: {deals.shape}")
print(f"\nДиапазон дат:")
print(f"  Created Time:  {deals['Created Time'].min()} — {deals['Created Time'].max()}")
print(f"  Closing Date:  {deals['Closing Date'].min()} — {deals['Closing Date'].max()}")
print(f"\nКонверсия (Won/Won+Lost): {(deals['Stage_Group']=='Won').sum() / ((deals['Stage_Group']=='Won').sum() + (deals['Stage_Group']=='Lost').sum()) * 100:.2f}%")
print(f"\nФинансы:")
print(f"  Общая выручка (sum Initial Amount Paid): €{deals['Initial Amount Paid'].sum():,.0f}")
print(f"  Сумма всех Offer Total: €{deals['Offer Total Amount'].sum():,.0f}")

ФИНАЛЬНАЯ СВОДКА — DEALS
Размер: (21593, 26)

Диапазон дат:
  Created Time:  2023-07-03 17:03:00 — 2024-06-21 15:30:00
  Closing Date:  2022-10-11 00:00:00 — 2024-12-11 00:00:00

Конверсия (Won/Won+Lost): 5.17%

Финансы:
  Общая выручка (sum Initial Amount Paid): €3,957,112
  Сумма всех Offer Total: €29,833,711


In [105]:
mask_invalid_order = (
    (deals['Closing Date'].dt.date < deals['Created Time'].dt.date) 
    & deals['Closing Date'].notna()
)

print(f"Обнаружено аномалий: {mask_invalid_order.sum()}")
print(deals.loc[mask_invalid_order, ['Id', 'Created Time', 'Closing Date', 'Stage', 'Deal Owner Name']])

Обнаружено аномалий: 44
                        Id        Created Time Closing Date         Stage  \
454    5805028000055502890 2024-06-16 00:06:00   2024-06-11          Lost   
2083   5805028000051847114 2024-05-25 21:29:00   2024-05-22          Lost   
2787   5805028000049539444 2024-05-12 11:19:00   2024-05-07          Lost   
3019   5805028000048886321 2024-05-08 15:31:00   2024-05-07  Payment Done   
3022   5805028000048883316 2024-05-08 14:48:00   2024-04-17          Lost   
3031   5805028000048886160 2024-05-08 12:54:00   2024-05-07  Payment Done   
3691   5805028000047482214 2024-04-30 15:16:00   2024-04-23          Lost   
4107   5805028000046234250 2024-04-24 17:30:00   2024-04-17          Lost   
4169   5805028000045961466 2024-04-23 21:44:00   2024-04-18          Lost   
4434   5805028000045314301 2024-04-21 08:57:00   2024-04-18          Lost   
4520   5805028000045245616 2024-04-20 06:19:00   2023-08-21          Lost   
4798   5805028000044499302 2024-04-17 09:10:00   202

### !!!! нашли 44 аномалии, где дата создания позже чем дата закрытия сделки!!!
и одна видимо опечатка - 2022-10-11 - за диапазонами датасета. 
Посмотрим что внутри по сделкам. 

In [109]:
anomaly_won = deals[
    deals['is_closing_date_anomaly'] & 
    (deals['Stage'] == 'Payment Done')
]
print(anomaly_won[['Id', 'Created Time', 'Closing Date', 'Months of study', 
                   'Initial Amount Paid', 'Offer Total Amount', 
                   'Product', 'Course duration']])

                        Id        Created Time Closing Date  Months of study  \
3019   5805028000048886321 2024-05-08 15:31:00   2024-05-07              2.0   
3031   5805028000048886160 2024-05-08 12:54:00   2024-05-07              1.0   
6357   5805028000042015392 2024-04-05 10:40:00   2023-10-03              1.0   
14007  5805028000022036007 2023-12-19 10:37:00   2023-07-01              NaN   

       Initial Amount Paid  Offer Total Amount            Product  \
3019                1000.0             11000.0       UX/UI Design   
3031                1000.0             11000.0  Digital Marketing   
6357                1000.0             11000.0       UX/UI Design   
14007                  0.0                 0.0                NaN   

       Course duration  
3019              11.0  
3031              11.0  
6357              11.0  
14007              NaN  


У нас 3 сделки оплачены и по ним идет обучение. 
### Решение - пометить эти 44 даты как аномальные, для дальнейшего исключения из расчета времени закрытия сделки и чтоб она не ушла в минус. Для юнит -экономики - ориентироваться по дате закрытия. 

In [110]:
deals['is_closing_date_anomaly'] = (
    (deals['Closing Date'].dt.date < deals['Created Time'].dt.date) 
    & deals['Closing Date'].notna()
)
print(f"Аномалий: {deals['is_closing_date_anomaly'].sum()}")

Аномалий: 44


# Сохраняем

In [123]:
deals.to_parquet(DATA_PROCESSED / 'deals_clean.parquet')
deals.to_excel(DATA_PROCESSED / 'deals_clean.xlsx', index=False)

print("✅ Сохранено!")
print(f"   {DATA_PROCESSED / 'deals_clean.parquet'}")
print(f"   {DATA_PROCESSED / 'deals_clean.xlsx'}")

✅ Сохранено!
   ..\data\processed\deals_clean.parquet
   ..\data\processed\deals_clean.xlsx
